# ANN with MLFlow

## Imports

In [52]:
import keras
import numpy as np
import pandas as pd
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split


import mlflow
from mlflow.models import infer_signature
 

In [53]:
data = pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";"
    )

In [54]:
train, test = train_test_split(data, test_size=0.25, random_state=42)

In [55]:
train

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
2835,6.3,0.25,0.22,3.30,0.048,41.0,161.0,0.99256,3.16,0.50,10.5,6
1157,7.8,0.30,0.29,16.85,0.054,23.0,135.0,0.99980,3.16,0.38,9.0,6
744,7.4,0.38,0.27,7.50,0.041,24.0,160.0,0.99535,3.17,0.43,10.0,5
1448,7.4,0.16,0.49,1.20,0.055,18.0,150.0,0.99170,3.23,0.47,11.2,6
3338,7.2,0.27,0.28,15.20,0.046,6.0,41.0,0.99665,3.17,0.39,10.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4426,6.2,0.21,0.52,6.50,0.047,28.0,123.0,0.99418,3.22,0.49,9.9,6
466,7.0,0.14,0.32,9.00,0.039,54.0,141.0,0.99560,3.22,0.43,9.4,6
3092,7.6,0.27,0.52,3.20,0.043,28.0,152.0,0.99129,3.02,0.53,11.4,6
3772,6.3,0.24,0.29,13.70,0.035,53.0,134.0,0.99567,3.17,0.38,10.6,6


In [56]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4898 entries, 0 to 4897
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         4898 non-null   float64
 1   volatile acidity      4898 non-null   float64
 2   citric acid           4898 non-null   float64
 3   residual sugar        4898 non-null   float64
 4   chlorides             4898 non-null   float64
 5   free sulfur dioxide   4898 non-null   float64
 6   total sulfur dioxide  4898 non-null   float64
 7   density               4898 non-null   float64
 8   pH                    4898 non-null   float64
 9   sulphates             4898 non-null   float64
 10  alcohol               4898 non-null   float64
 11  quality               4898 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 459.3 KB


In [57]:
train_x=train.drop(columns=['quality']).values
train_y=train[['quality']].values.ravel()

In [58]:
test_x=test.drop(columns=['quality']).values
test_y=test[['quality']].values.ravel()

In [59]:
train_x, valid_x, train_y, valid_y = train_test_split(train_x, train_y, test_size=0.20, random_state=42)
signature = infer_signature(train_x, train_y)

In [60]:
### ANN Model
def train_model(params, epochs, train_x, train_y, valid_x, valid_y, test_x, test_y):

    ## define the model architecture
    mean = np.mean(train_x, axis=0)
    var = np.var(train_x, axis=0)
    model=keras.Sequential(
        [
            keras.Input([train_x.shape[1]]),
            keras.layers.Normalization(mean=mean, variance=var),
            keras.layers.Dense(64, activation='relu'),
            keras.layers.Dense(1)
        ]
        )
    model.compile(optimizer=keras.optimizers.SGD(
        learning_rate=params["lr"], momentum=params["momentum"]
        ),
        loss="mean_squared_error",
        metrics=[keras.metrics.RootMeanSquaredError()]
        )

    ## train the model with lr and momentum params with mlflow tracking
    with mlflow.start_run(nested=True):
        model.fit(train_x, train_y, validation_data=(valid_x, valid_y),
                  epochs=epochs,
                  batch_size=64)

        eval_result=model.evaluate(valid_x, valid_y, batch_size=64)
        eval_rmse=eval_result[1]

        # log params and result
        mlflow.log_params(params)
        mlflow.log_metric("eval_rmse", eval_rmse)
        # log model
        mlflow.tensorflow.log_model(model, "model", signature=signature)
        return {"loss":eval_rmse, "status":STATUS_OK, "model":model}
        

In [61]:
def objective(params):
    result = train_model(
        params,
        epochs=3,
        train_x=train_x,
        train_y=train_y,
        valid_x=valid_x,
        valid_y=valid_y,
        test_x=test_x,
        test_y=test_y,
        )
    return result


In [62]:
space ={
    "lr": hp.loguniform("lr",np.log(1e-5),np.log(1e-1)),
    "momentum":hp.uniform("momentum",0.0,1.0) 
    }

In [63]:
print("X_train:", train_x.shape)
print("y_train:", train_y.shape)

print("X_val:", valid_x.shape)
print("y_val:", valid_y.shape)

X_train: (2938, 11)
y_train: (2938,)
X_val: (735, 11)
y_val: (735,)


In [64]:
mlflow.set_experiment("/wine-quality")

with mlflow.start_run():

    trials = Trials()

    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=4,
        trials=trials
    )

    best_run = sorted(
        trials.results,
        key=lambda x: x["loss"]
    )[0]

    # Log parameters and result
    mlflow.log_params(best)

    mlflow.log_metric(
        "eval_rmse",
        best_run["loss"]
    )

    # Log model
    mlflow.tensorflow.log_model(
        best_run["model"],
        name="model",
        signature=signature
    )

    print(f"Best parameters: {best}")
    print(f"Best eval RMSE: {best_run['loss']}")

Epoch 1/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 8s 190ms/step - loss: 34.8696 - root_mean_squared_error: 5.9050
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 5.8379 - root_mean_squared_error: 2.4162 - val_loss: 1.6866 - val_root_mean_squared_error: 1.2987

Epoch 2/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 1.5160 - root_mean_squared_error: 1.2312
  0%|          | 0/4 [00:00<?, ?trial/s, best loss=?]

error: operand #0 does not dominate this use
E0000 00:00:1788444243.301493 20121235 meta_optimizer.cc:967] tfg_optimizer{any(tfg-consolidate-attrs,tfg-toposort,tfg-shape-inference{graph-version=0},tfg-prepare-attrs-export)} failed: INVALID_ARGUMENT: MLIR Graph Optimizer failed: 
W0000 00:00:1788444243.301919 20121235 optimize_function_graph_utils.cc:634] Ignoring multi-device function optimization failure: INVALID_ARGUMENT: MLIR Graph Optimizer failed: 
error: operand #0 does not dominate this use
W0000 00:00:1788444243.414474 20121235 optimize_function_graph_utils.cc:634] Ignoring multi-device function optimization failure: INVALID_ARGUMENT: MLIR Graph Optimizer failed: 


46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.3242 - root_mean_squared_error: 1.1508 - val_loss: 1.2638 - val_root_mean_squared_error: 1.1242

Epoch 3/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.9140 - root_mean_squared_error: 0.9560
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.0418 - root_mean_squared_error: 1.0207 - val_loss: 1.0427 - val_root_mean_squared_error: 1.0211

 1/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.9294 - root_mean_squared_error: 0.9640
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.0427 - root_mean_squared_error: 1.0211

  0%|          | 0/4 [00:00<?, ?trial/s, best loss=?]

2026/09/03 17:34:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 7s 169ms/step - loss: 43.1978 - root_mean_squared_error: 6.5725
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3.7213 - root_mean_squared_error: 1.9291 - val_loss: 1.0706 - val_root_mean_squared_error: 1.0347

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.9897 - root_mean_squared_error: 0.9948
 25%|██▌       | 1/4 [00:07<00:21,  7.23s/trial, best loss: 1.0211211442947388]

...

Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 7s 157ms/step - loss: 35.0983 - root_mean_squared_error: 5.9244
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 35.9665 - root_mean_squared_error: 5.9972 - val_loss: 35.7327 - val_root_mean_squared_error: 5.9777

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 32.5316 - root_mean_squared_error: 5.7036
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 35.0632 - root_mean_squared_error: 5.9214 - val_loss: 34.8356 - val_root_mean_squared_error: 5.9022

Epoch 3/3                                                                      

 75%|███████▌  | 3/4 [00:20<00:06,  6.72s/trial, best loss: 0.8232943415641785]

error: operand #0 does not dominate this use
W0000 00:00:1788444263.678237 20121235 optimize_function_graph_utils.cc:634] Ignoring multi-device function optimization failure: INVALID_ARGUMENT: MLIR Graph Optimizer failed: 
error: operand #0 does not dominate this use
W0000 00:00:1788444263.782191 20121235 optimize_function_graph_utils.cc:634] Ignoring multi-device function optimization failure: INVALID_ARGUMENT: MLIR Graph Optimizer failed: 


 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 34.5010 - root_mean_squared_error: 5.8738
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 34.1852 - root_mean_squared_error: 5.8468 - val_loss: 33.9631 - val_root_mean_squared_error: 5.8278

 1/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 33.5339 - root_mean_squared_error: 5.7908
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 33.9631 - root_mean_squared_error: 5.8278

 75%|███████▌  | 3/4 [00:20<00:06,  6.72s/trial, best loss: 0.8232943415641785]

2026/09/03 17:34:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



100%|██████████| 4/4 [00:26<00:00,  6.50s/trial, best loss: 0.8232943415641785]
Best parameters: {'lr': np.float64(0.015440308064356685), 'momentum': np.float64(0.4548074712508233)}
Best eval RMSE: 0.8232943415641785
